# 6.8 · 主成分分析 / Principal Component Analysis (PCA)

> **课程定位 / Where this fits**
> 第 8 课，**Part 6 · 无监督学习**。
> Lesson 8, **Part 6 · Unsupervised Learning**.
>
> 前 7 课讲聚类（发现分组），这一课转入**降维**（发现紧凑表示）。PCA 是最经典、最重要的线性降维：找数据中**方差最大的正交方向（主成分）**，用最少的维度保留最多的信息。它无处不在——可视化、去噪、压缩、加速下游模型、缓解维度灾难(5.3)。
> The first 7 lessons did clustering (finding groups); now we turn to **dimensionality reduction** (finding compact representations). PCA is the most classic, important linear DR: find the **orthogonal directions of maximum variance (principal components)** and keep the most information in the fewest dimensions. It's everywhere — visualization, denoising, compression, speeding up downstream models, easing the curse of dimensionality (5.3).

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $\mathbf{X}$ —— 数据矩阵（已中心化）/ data matrix (centered)
> - $\mathbf{C}=\frac1n\mathbf{X}^\top\mathbf{X}$ —— 协方差矩阵 / covariance matrix
> - $\mathbf{w}_k$ —— 第 $k$ 个主成分（单位向量）/ the $k$-th principal component
> - $\lambda_k$ —— 第 $k$ 个特征值（= 该方向的方差）/ the $k$-th eigenvalue (= variance along it)

> 💡 **面试相关 / Interview-relevant**
> - "PCA 在做什么 / 主成分是什么"（★★★★★）
> - "PCA 与协方差矩阵特征分解 / SVD 的关系"（★★★★★）
> - "为什么 PCA 前要中心化（和标准化）"（★★★★★）
> - "怎么选主成分个数（解释方差）"（★★★★）
> - "PCA vs LDA" / "PCA 的假设与局限"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解 PCA 的两种等价视角：最大方差 / 最小重构误差。
   Understand PCA's two equivalent views: max variance / min reconstruction error.
2. 用协方差特征分解 / SVD **从零**实现。
   Implement **from scratch** via covariance eigen-decomposition / SVD.
3. 理解中心化与标准化的必要性。
   Understand why centering and scaling matter.
4. 用解释方差比选维度 + 做重构/去噪。
   Choose dimensions by explained variance, and do reconstruction/denoising.
5. 知道 PCA 的假设与局限（引出核 PCA 6.9）。
   Know PCA's assumptions and limits (motivating Kernel PCA, 6.9).

## 目录 / TOC
1. [先建直觉：把信息压进少数方向](#1)
2. [两种视角 + 数学 ⭐](#2)
3. [🌸 数据：Iris + 从零(SVD) ⭐](#3)
4. [中心化 / 标准化 ⭐](#4)
5. [解释方差选维度 ⭐](#5)
6. [🔢 重构与去噪：Digits](#6)
7. [假设与局限 + 小结](#7)


<a id="1"></a>
## 1. 先建直觉：把信息压进少数方向 / Intuition First

想象一团点在三维空间里其实近似躺在**一张斜放的纸**上——虽然记了 3 个坐标，但真正的"信息"只有 2 维（纸面上的位置），第三维几乎没变化。PCA 就是自动找到这张"纸"：**用尽量少的方向，抓住数据变化最大的那几个方向。**
Imagine a cloud of points in 3-D that actually lies almost flat on **a tilted sheet of paper** — we store 3 coordinates, but the real "information" is 2-D (position on the sheet); the third barely varies. PCA automatically finds that sheet: **capture the few directions along which the data varies most, using as few of them as possible.**

为什么找"方差最大"的方向？因为**方差大 = 数据在这个方向上差异大 = 这个方向携带的信息多**；方差几乎为 0 的方向（如纸的厚度）丢掉也不可惜。
Why the "max variance" directions? Because **large variance = points differ a lot along it = it carries more information**; a near-zero-variance direction (like the paper's thickness) can be dropped harmlessly.

主成分还要求**彼此正交**（互不重复），并按方差从大到小排序。取前几个，就把高维数据压成了低维。
The components are also required to be **orthogonal** (non-redundant), ordered by decreasing variance. Keep the first few and you've compressed high-dim data into low-dim.


<a id="2"></a>
## 2. 两种视角 + 数学 ⭐ / Two Views & Math

PCA 有两个等价的定义（导出同一答案）：
PCA has two equivalent definitions (yielding the same answer):
- **最大方差 / max variance**：找一个单位方向 $\mathbf{w}$，使数据投影 $\mathbf{Xw}$ 的方差最大；第二主成分在与第一正交的约束下方差最大；以此类推。
  Find a unit direction $\mathbf{w}$ maximizing the variance of the projection $\mathbf{Xw}$; the second component maximizes variance subject to being orthogonal to the first; and so on.
- **最小重构误差 / min reconstruction error**：找一个 $k$ 维子空间，使所有点投影到它上面的误差（平方和）最小。
  Find a $k$-dim subspace minimizing the squared projection error of all points.

**数学解**：设数据已中心化（每列减去均值），协方差矩阵 $\mathbf{C}=\frac1n\mathbf{X}^\top\mathbf{X}$。最大化 $\mathbf{w}^\top\mathbf{C}\mathbf{w}$（在 $\|\mathbf{w}\|=1$ 约束下），用拉格朗日乘子可得：
**Solution:** with centered data, covariance $\mathbf{C}=\frac1n\mathbf{X}^\top\mathbf{X}$. Maximizing $\mathbf{w}^\top\mathbf{C}\mathbf{w}$ subject to $\|\mathbf{w}\|=1$, via Lagrange multipliers, gives:

$$\mathbf{C}\mathbf{w} = \lambda\mathbf{w}$$

也就是说，**主成分 = 协方差矩阵的特征向量**，对应特征值 $\lambda$ = 该方向上的方差。按 $\lambda$ 从大到小取前 $k$ 个特征向量即可。
That is, **principal components = eigenvectors of the covariance matrix**, with eigenvalue $\lambda$ = the variance along that direction. Take the top-$k$ eigenvectors by $\lambda$.

**SVD 视角**（数值上更稳，sklearn 用它）：对 $\mathbf{X}=\mathbf{U}\boldsymbol\Sigma\mathbf{V}^\top$ 做奇异值分解，则 $\mathbf{V}$ 的列就是主成分，奇异值的平方 $\propto$ 特征值。
**SVD view** (numerically more stable; sklearn uses it): for $\mathbf{X}=\mathbf{U}\boldsymbol\Sigma\mathbf{V}^\top$, the columns of $\mathbf{V}$ are the principal components, and squared singular values $\propto$ eigenvalues.


<a id="3"></a>
## 3. 数据：Iris + 从零(SVD) ⭐ / Iris & From Scratch

先用 **Iris**（4 维 → 2 维）做可视化，从零用 SVD 实现并对照 sklearn。
We use **Iris** (4-D → 2-D) for visualization, implementing from scratch via SVD and comparing to sklearn.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=3, suppress=True)

iris = load_iris(); X, y = iris.data, iris.target
Xc = X - X.mean(0)                      # 中心化: 每列减去该列均值(PCA 必须的一步, 见第4节)

# 从零用 SVD 求主成分 / PCA via SVD from scratch
# X = U·diag(S)·Vt; Vt 的每一行是一个主成分(右奇异向量), S 是奇异值
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
PC = Vt[:2]                             # 取前两个主成分(方差最大的两个方向)
Z_scratch = Xc @ PC.T                   # 把数据投影到这两个方向上 → 得到 2 维坐标
explained = (S**2) / (S**2).sum()       # 解释方差比 = 奇异值² 占总和的比例

from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(Xc)
Z_sk = pca.transform(Xc)
print(f"从零解释方差比(前2) explained: {explained[:2]}  累计 cumulative {explained[:2].sum():.3f}")
print(f"sklearn 解释方差比 explained:  {pca.explained_variance_ratio_}")
# 主成分方向可能整体反号(±都对), 所以比较绝对值
print(f"两者投影一致(符号可能相反) match (sign may flip): {np.allclose(np.abs(Z_scratch), np.abs(Z_sk), atol=1e-6)}")

fig, ax = plt.subplots(figsize=(6.5,5))
for k,name in enumerate(iris.target_names):
    ax.scatter(Z_scratch[y==k,0], Z_scratch[y==k,1], label=name, s=25)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend()
ax.set_title(f"Iris PCA 4D→2D: 前2主成分保留 {explained[:2].sum():.0%} 方差 / top-2 PCs keep {explained[:2].sum():.0%} variance")
plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. 中心化 / 标准化 ⭐ / Centering & Scaling

- **中心化（减均值）是必须的**：协方差/方差的定义本身就基于"偏离均值"。如果不中心化，第一主成分会指向数据的"平均位置"而不是变化方向。sklearn 的 PCA **会自动中心化**。
  **Centering (subtract the mean) is mandatory:** variance/covariance are defined around the mean. Without it, the first component points toward the data's average position rather than its direction of variation. sklearn's PCA **centers automatically**.
- **标准化（再除以标准差）视情况而定**：PCA 看方差，若特征量纲悬殊（如收入 vs 年龄），大量纲特征会主导主成分。量纲不一时应先标准化（等价于用**相关矩阵**而非协方差矩阵做 PCA）。
  **Scaling (then divide by std) depends:** PCA chases variance, so if scales differ wildly (income vs age), the large-scale feature dominates. Standardize first when scales differ (equivalent to PCA on the **correlation matrix** rather than covariance).


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_wine
wine = load_wine()
Xw = wine.data
# 不标准化: proline(~1000) 这种大量纲特征会几乎独占第一主成分
pca_raw = PCA(2).fit(Xw - Xw.mean(0))                    # 只中心化
pca_std = PCA(2).fit(StandardScaler().fit_transform(Xw)) # 中心化 + 除以标准差
print("Wine 数据(特征量纲差异大, 如 proline~1000 vs flavanoids~3):")
print(f"  不标准化 PC1 解释方差比 unscaled: {pca_raw.explained_variance_ratio_[0]:.3f} (被大量纲特征绑架 hijacked)")
print(f"  标准化后 PC1 解释方差比 scaled:   {pca_std.explained_variance_ratio_[0]:.3f} (更均衡 balanced)")
print("\n量纲悬殊时先标准化(=用相关矩阵); 同量纲(如像素)可只中心化 / scale when units differ")


<a id="5"></a>
## 5. 解释方差选维度 ⭐ / Choosing Components

每个主成分的**解释方差比** = $\lambda_k / \sum\lambda$（它解释了总方差的百分之几）。常用做法：画**累计解释方差**曲线（碎石图 scree plot），取"保留 90%/95% 方差"所需的维度。
Each component's **explained variance ratio** = $\lambda_k / \sum\lambda$ (what fraction of total variance it explains). Common practice: plot the **cumulative explained variance** (scree plot) and take the number of dimensions needed to keep 90%/95% of the variance.


In [ ]:
pca_full = PCA().fit(StandardScaler().fit_transform(Xw))   # 不限维度, 算出全部主成分
cum = np.cumsum(pca_full.explained_variance_ratio_)         # 累计解释方差(逐个累加)
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(range(1, len(cum)+1), pca_full.explained_variance_ratio_, alpha=0.5, label="单个 individual")
ax.plot(range(1, len(cum)+1), cum, "o-", color="red", label="累计 cumulative")
ax.axhline(0.95, color="gray", ls="--", label="95% 阈值 threshold")
k95 = np.argmax(cum >= 0.95) + 1            # 第一个累计≥95% 的位置(+1 因下标从0起)
ax.axvline(k95, color="green", ls=":", label=f"{k95} 维达 95%")
ax.set_xlabel("主成分数 #components"); ax.set_ylabel("解释方差比 explained variance"); ax.legend()
ax.set_title(f"Scree plot: Wine 13 维 → {k95} 维即保留 95% 方差 / 13→{k95} dims keep 95%")
plt.tight_layout(); plt.show()
print(f"Wine: {k95} 个主成分保留 95% 方差; sklearn 可直接写 PCA(n_components=0.95) 自动选")


<a id="6"></a>
## 6. 重构与去噪：Digits / Reconstruction & Denoising

PCA 可以**逆变换**（近似重构原数据）：用前 $k$ 个主成分重建，丢掉的是小方差方向——而噪声往往就藏在那些小方差方向里。这给了 PCA 一个**去噪/压缩**的用途。用 **Digits**（64 维像素）演示。
PCA can **inverse-transform** (approximately reconstruct): rebuild from the top-$k$ components, discarding small-variance directions — where noise often hides. This gives PCA a **denoising/compression** use. Demoed on **Digits** (64-D pixels).


In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
Xd = digits.data
rng = np.random.default_rng(0)
Xn = Xd + rng.normal(0, 4, Xd.shape)     # 给图像加高斯噪声 / add Gaussian noise

pca_d = PCA(n_components=20).fit(Xd)      # 在干净数据上学 64→20 维的投影
# transform 把噪声图投到 20 维(丢掉小方差方向=噪声), inverse_transform 再重构回 64 维
Xden = pca_d.inverse_transform(pca_d.transform(Xn))
print(f"Digits 64维 → 20 主成分保留方差 variance kept: {pca_d.explained_variance_ratio_.sum():.2%}")

fig, axes = plt.subplots(3, 8, figsize=(11, 4.2))
for j in range(8):
    axes[0,j].imshow(Xd[j].reshape(8,8), cmap="gray_r"); axes[0,j].axis("off")   # 原始
    axes[1,j].imshow(Xn[j].reshape(8,8), cmap="gray_r"); axes[1,j].axis("off")   # 加噪
    axes[2,j].imshow(Xden[j].reshape(8,8), cmap="gray_r"); axes[2,j].axis("off") # 去噪
for yv,t in zip([0.78,0.5,0.22], ["原始 orig","加噪 noisy","去噪 denoised"]):
    fig.text(0.08, yv, t, va="center")
plt.suptitle("PCA 去噪: 投影到前20主成分再重构, 丢掉的小方差方向多是噪声 / denoise by dropping small-variance dirs")
plt.tight_layout(); plt.show()


<a id="7"></a>
## 7. 假设与局限 + 小结 / Assumptions, Limits & Summary

PCA 的假设与局限：
PCA's assumptions and limits:
- **线性 / linear**：只找线性子空间。数据在**弯曲流形**上（如瑞士卷）时无能为力 → 核 PCA(6.9)/t-SNE(6.12)/UMAP(6.13)。
  Only finds linear subspaces; fails on **curved manifolds** (Swiss roll) → Kernel PCA/t-SNE/UMAP.
- **方差 ≠ 重要性 / variance ≠ importance**：PCA 假设大方差方向更重要，但有时判别信息藏在小方差方向里（此时 LDA 5.12/6.14 更合适——它用标签找可分方向）。
  PCA assumes high-variance = important, but discriminative info can hide in low-variance directions (then LDA, which uses labels, is better).
- **对异常值敏感 / outlier-sensitive**：方差会被离群点拉偏。
  Variance is skewed by outliers.

```
PCA: 找方差最大的正交方向(主成分)= 协方差矩阵特征向量(或 X 的 SVD 右奇异向量)
两视角等价: 最大化投影方差 = 最小化重构误差
中心化必须(sklearn 自动); 量纲悬殊先标准化(=相关矩阵 PCA)
选维度: 累计解释方差(95%) / scree plot; PCA(n_components=0.95)
用途: 可视化/压缩/去噪(逆变换丢小方差方向)/加速下游/缓解维度灾难
局限: 线性、方差≠重要、对异常敏感 → 非线性用核PCA/t-SNE/UMAP
```

### 💡 面试速查 / Interview cheat-sheet
1. **主成分 = 协方差矩阵特征向量**（按特征值降序）；特征值 = 该方向方差。
   PCs = covariance eigenvectors (by descending eigenvalue); eigenvalue = variance along it.
2. **SVD 数值更稳**（sklearn 用）；最大方差 ⟺ 最小重构误差。
   SVD is more stable (sklearn uses it); max variance ⟺ min reconstruction error.
3. **必须中心化**；量纲不一**先标准化**。
   Must center; standardize first when scales differ.
4. **选维度**看累计解释方差（常 90/95%）。
   Choose dims by cumulative explained variance (often 90/95%).
5. **PCA(无监督,方差) vs LDA(有监督,可分性)**；局限是线性 → 核 PCA。
   PCA (unsupervised, variance) vs LDA (supervised, separability); linear limit → Kernel PCA.

### 下一节 / Next
**6.9 核 PCA**——用核技巧(5.5 见过)把 PCA 推广到非线性，能"展开"瑞士卷这类弯曲流形。
**6.9 Kernel PCA** — uses the kernel trick (seen in 5.5) to make PCA nonlinear, able to "unroll" curved manifolds like the Swiss roll.
